# Mineral Trade Data Exploration
## Plutus Hackathon 2025 - Critical Mineral Forecasting

This notebook explores the EXIM trade data for Copper, Lithium, and Graphite.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

# Load data
trade_df = pd.read_csv('../data/processed/trade_data.csv', parse_dates=['date'])
prices_df = pd.read_csv('../data/processed/global_prices.csv', parse_dates=['date'])
production_df = pd.read_csv('../data/processed/domestic_production.csv', parse_dates=['date'])

print(f"Trade data: {len(trade_df):,} records")
print(f"Date range: {trade_df['date'].min()} to {trade_df['date'].max()}")

In [ ]:
# Summary statistics by mineral
summary = trade_df.groupby(['mineral', 'flow']).agg({
    'value_usd_mn': ['sum', 'mean', 'std'],
    'quantity_mt': ['sum', 'mean']
}).round(2)
summary

In [ ]:
# Time series plot
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for i, mineral in enumerate(['Copper', 'Lithium', 'Graphite']):
    mineral_data = trade_df[trade_df['mineral'] == mineral]
    imports = mineral_data[mineral_data['flow'] == 'Import'].groupby('date')['value_usd_mn'].sum()
    exports = mineral_data[mineral_data['flow'] == 'Export'].groupby('date')['value_usd_mn'].sum()
    
    axes[i].plot(imports.index, imports.values, label='Imports', color='#EF4444')
    axes[i].plot(exports.index, exports.values, label='Exports', color='#10B981')
    axes[i].set_title(f'{mineral} Trade Trends (USD Million)', fontsize=12, fontweight='bold')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/trade_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Import dependency calculation
for mineral in ['Copper', 'Lithium', 'Graphite']:
    imports = trade_df[(trade_df['mineral'] == mineral) & (trade_df['flow'] == 'Import')]['quantity_mt'].sum()
    production = production_df[production_df['mineral'] == mineral]['production_mt'].sum()
    total_supply = imports + production
    dependency = imports / total_supply if total_supply > 0 else 1.0
    print(f"{mineral}: Import Dependency = {dependency:.1%}")

In [ ]:
# Top trading partners
top_partners = trade_df[trade_df['flow'] == 'Import'].groupby(['mineral', 'country'])['value_usd_mn'].sum()
top_partners = top_partners.reset_index().sort_values(['mineral', 'value_usd_mn'], ascending=[True, False])
top_partners.groupby('mineral').head(5)

In [ ]:
# HHI Concentration Index
def calculate_hhi(df):
    total = df['value_usd_mn'].sum()
    shares = df['value_usd_mn'] / total
    return (shares ** 2).sum()

for mineral in ['Copper', 'Lithium', 'Graphite']:
    mineral_imports = trade_df[(trade_df['mineral'] == mineral) & (trade_df['flow'] == 'Import')]
    country_totals = mineral_imports.groupby('country')['value_usd_mn'].sum().reset_index()
    hhi = calculate_hhi(country_totals)
    concentration = 'High' if hhi > 0.25 else 'Medium' if hhi > 0.15 else 'Low'
    print(f"{mineral}: HHI = {hhi:.4f} ({concentration} concentration)")

## ANOVA Analysis
Compare trade performance across mineral categories

In [ ]:
# One-way ANOVA: Trade values by mineral
from scipy.stats import f_oneway

copper_values = trade_df[trade_df['mineral'] == 'Copper']['value_usd_mn'].values
lithium_values = trade_df[trade_df['mineral'] == 'Lithium']['value_usd_mn'].values
graphite_values = trade_df[trade_df['mineral'] == 'Graphite']['value_usd_mn'].values

f_stat, p_value = f_oneway(copper_values, lithium_values, graphite_values)
print(f"ANOVA Results:")
print(f"F-statistic: {f_stat:.2f}")
print(f"P-value: {p_value:.6f}")
print(f"Significant: {'Yes' if p_value < 0.05 else 'No'}")